In [0]:
src = dp["src_catalog"]

ctas = (spark.table(f"{src}.bcp_udv_int_vu.h_cuentafinanciera")
        .select(F.trim(F.col("CODCLAVEPARTYCLI")).alias("codclavepartycli"),
                F.trim(F.col("CODCLAVECTA")).alias("codclavecta")))   # SIN filtro de CODAPP

tc = (spark.table(f"{src}.bcp_udv_int_vu.h_saldocuentatarjetacredito")
      .filter(F.col("CODMES")==202512)
      .select(F.trim(F.col("CODCLAVECTA")).alias("codclavecta"), F.col("CTDDIAMOROSO").alias("dias")))
cp = (spark.table(f"{src}.bcp_udv_int_vu.h_saldocuentacreditopersonal")
      .filter(F.col("CODMES")==202512)
      .select(F.trim(F.col("CODCLAVECTA")).alias("codclavecta"), F.col("CTDDIAVCDA").alias("dias")))

raw = (tc.unionByName(cp).join(ctas, "codclavecta")
       .groupBy("codclavepartycli").agg(F.max("dias").alias("max_raw")))

dif.join(raw, "codclavepartycli", "left").select(
    F.count("*").alias("n_dif"),
    F.sum(F.when(F.col("max_raw")==F.col("arnold"),1).otherwise(0)).alias("match_raw"),
    F.sum(F.when(F.col("max_raw")> F.col("arnold"),1).otherwise(0)).alias("raw_mayor"),
    F.sum(F.when(F.col("max_raw")< F.col("arnold"),1).otherwise(0)).alias("raw_menor"),
    F.sum(F.when(F.col("max_raw").isNull(),1).otherwise(0)).alias("sin_fila"),
).show()

In [0]:
tablas = [
  "catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizexperienciacliente",
  "catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_conceptodeudorrcc",
  "catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizdeudorrccproducto",
  path_porto,
  f"{src}.bcp_udv_int_vu.h_cuentafinanciera",
]
for t in tablas:
    cols = [c for c in spark.table(t).columns
            if any(k in c.lower() for k in ("atraso","mora","diamor","vcda"))]
    print(t, "->", cols)